# FullStack AI · Apprentissage automatique

**Objectif :** Prétraitement, régressions, régularisation, arbres, boosting, séries temporelles et clustering.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Le vocabulaire du machine learning

Le **machine learning** consiste à faire apprendre à un programme une règle à partir d'exemples, au lieu de l'écrire à la main.

## Les grandes familles

- **Apprentissage supervisé** : on dispose d'exemples avec la bonne réponse (la **cible**, ou *target*, notée y).
  - **Régression** : la cible est un nombre (un prix, une masse, une durée).
  - **Classification** : la cible est une catégorie (spam / non spam, espèce, survie).
- **Apprentissage non supervisé** : pas de cible. On cherche une structure : des groupes (*clustering*), une réduction de dimensions.
- L'**IA générative** produit du contenu (texte, images, code). Les grands modèles de langage sont entraînés de façon auto-supervisée : prédire le mot suivant.

## Features et cible

Les **features** (caractéristiques, notées X) sont les colonnes qui servent à prédire. La **cible** y est la colonne à prédire.

In [ ]:
import pandas as pd
df = pd.read_csv("penguins.csv").dropna()
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]]   # matrice : majuscule
y = df["species"]                                                   # vecteur : minuscule
print(X.shape, y.shape)

## Séparer entraînement et test

*Animation interactive « split » : à voir sur la plateforme web.*

Un modèle doit être évalué sur des données qu'il **n'a jamais vues**, sinon on mesure sa mémoire et non sa capacité à généraliser.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "exemples d'entraînement,", len(X_test), "de test")

`random_state` rend le découpage reproductible. `stratify=y` garde les mêmes proportions de classes dans les deux parties.

## Sous-apprentissage et surapprentissage

*Animation interactive « overfit » : à voir sur la plateforme web.*

- **Sous-apprentissage** (*underfitting*) : le modèle est trop simple. Il se trompe autant à l'entraînement qu'au test.
- **Surapprentissage** (*overfitting*) : le modèle a appris par cœur les données d'entraînement, bruit compris. Excellent à l'entraînement, mauvais au test.
- Le bon modèle se situe entre les deux : c'est le **compromis biais-variance**.

## Toujours une baseline

Avant tout modèle, calcule le score d'une stratégie naïve : prédire la classe majoritaire, ou la moyenne. Un modèle qui ne fait pas mieux ne sert à rien.

In [ ]:
print(y_train.value_counts(normalize=True).round(3))

> **À retenir** : features X, cible y, séparation train/test, baseline. Ces quatre réflexes s'appliquent à tous les projets, y compris en deep learning.

### Exercice 1 · Préparer X et y

Avec `tips.csv`, on veut prédire le pourboire. Crée `X` (colonnes `total_bill` et `size`) et `y` (colonne `tip`), puis sépare-les avec `train_test_split` (25 % de test, `random_state=0`) en `X_train, X_test, y_train, y_test`.

<details><summary>Indice</summary>

X avec des doubles crochets, y avec des simples. train_test_split(X, y, test_size=0.25, random_state=0).
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
tips = pd.read_csv("tips.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert list(X.columns) == ["total_bill", "size"], f"Colonnes de X : {list(X.columns)}."
assert X_train.shape == (183, 2) and X_test.shape == (61, 2), f"Formes : {X_train.shape} et {X_test.shape}, j'attendais (183, 2) et (61, 2)."
assert len(y_train) == 183, "y_train doit avoir 183 valeurs."
print("Bravo, tout est correct.")

### Exercice 2 · Le score à battre

Sur le Titanic, calcule `baseline` : l'exactitude (*accuracy*) obtenue en prédisant **toujours** la classe la plus fréquente de `survived`. Exprime-la entre 0 et 1.

<details><summary>Indice</summary>

value_counts(normalize=True) donne les proportions ; la plus grande est l'exactitude de la stratégie « toujours la classe majoritaire ».
</details>

In [ ]:
import pandas as pd
t = pd.read_csv("titanic.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(baseline - 0.6162) < 0.001, f"baseline vaut {baseline}, j'attendais environ 0.616."
print("Bravo, tout est correct.")

---
## Classification et métriques

La **régression logistique** est le modèle de classification de base. Malgré son nom, elle classe : elle calcule une combinaison linéaire des features, puis la transforme en probabilité entre 0 et 1 grâce à la fonction **sigmoïde**.

$$
P(y = 1 \mid x) = \sigma(w \cdot x + b) = \frac{1}{1 + e^{-(w \cdot x + b)}}
$$

## Préparer les données

Les modèles ne comprennent que des nombres. Il faut **encoder** les catégories et **imputer** les valeurs manquantes.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)          # encodage binaire
t["age"] = t["age"].fillna(t["age"].median())
features = ["pclass", "femme", "age", "fare", "sibsp", "parch"]
X, y = t[features], t["survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]       # probabilité de survie
print(pred[:5], proba[:5].round(2))

Pour une variable à plusieurs catégories (comme `embarked`), on utilise le **one-hot encoding** : `pd.get_dummies(t["embarked"])` crée une colonne 0/1 par catégorie.

## La matrice de confusion

In [ ]:
from sklearn.metrics import confusion_matrix
pd.DataFrame(confusion_matrix(y_test, pred),
             index=["réel : décédé", "réel : survivant"],
             columns=["prédit : décédé", "prédit : survivant"])

- **Vrais positifs (VP)** : survivants prédits survivants. **Faux positifs (FP)** : décédés prédits survivants.
- **Vrais négatifs (VN)** et **faux négatifs (FN)** : de même pour l'autre classe.

## Les métriques

$$
\text{précision} = \frac{VP}{VP + FP} \qquad \text{rappel} = \frac{VP}{VP + FN} \qquad F_1 = 2 \cdot \frac{\text{précision} \cdot \text{rappel}}{\text{précision} + \text{rappel}}
$$

- **Exactitude** (*accuracy*) : la part de bonnes réponses. Trompeuse si les classes sont déséquilibrées.
- **Précision** : parmi les prédits positifs, combien le sont vraiment ? Importante quand un faux positif coûte cher (un mail légitime classé spam).
- **Rappel** (*recall*) : parmi les vrais positifs, combien en a-t-on trouvé ? Important quand un faux négatif coûte cher (une fraude ou une maladie non détectée).

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print(f"exactitude {accuracy_score(y_test, pred):.3f}  précision {precision_score(y_test, pred):.3f}  "
      f"rappel {recall_score(y_test, pred):.3f}  F1 {f1_score(y_test, pred):.3f}")

> **À retenir** : choisis ta métrique en fonction du **coût des erreurs** dans le problème métier, et dis-le explicitement dans tes projets.

### Exercice 1 · Ajouter le port d'embarquement

Améliore le modèle de la leçon : ajoute un encodage one-hot de `embarked` (après avoir remplacé les valeurs manquantes par `"S"`). Construis `X` avec les features de base plus ces colonnes, entraîne `clf` (même split que la leçon) et calcule `acc`, l'exactitude sur le test. Elle doit dépasser 0.78.

<details><summary>Indice</summary>

pd.get_dummies(t["embarked"].fillna("S"), prefix="port", dtype=int), puis pd.concat([t[base], ports], axis=1).
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)
t["age"] = t["age"].fillna(t["age"].median())
base = ["pclass", "femme", "age", "fare", "sibsp", "parch"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert X.shape[1] == 9, f"X a {X.shape[1]} colonnes, j'en attendais 9 (6 de base + 3 ports)."
assert X.isna().sum().sum() == 0, "X contient encore des valeurs manquantes."
assert acc > 0.78, f"acc vaut {acc:.3f}, vise plus de 0.78."
print("Bravo, tout est correct.")

### Exercice 2 · Précision et rappel à la main

À partir de la matrice de confusion ci-dessous (lignes = réel, colonnes = prédit, ordre [négatif, positif]), calcule `precision`, `rappel` et `f1` **sans scikit-learn**.

<details><summary>Indice</summary>

cm.ravel() donne [VN, FP, FN, VP]. Applique ensuite les formules.
</details>

In [ ]:
import numpy as np
cm = np.array([[95, 15],
               [20, 49]])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(precision - 49 / 64) < 1e-9, f"precision vaut {precision}, j'attendais 49/64 ≈ 0.766."
assert abs(rappel - 49 / 69) < 1e-9, f"rappel vaut {rappel}, j'attendais 49/69 ≈ 0.710."
assert abs(f1 - 0.7368) < 1e-3, f"f1 vaut {f1}."
print("Bravo, tout est correct.")

---
## Prétraitement avancé : ColumnTransformer et pipelines

Dans un vrai jeu de données, les colonnes n'ont pas toutes besoin du même traitement : les nombres doivent être imputés et mis à l'échelle, les catégories encodées. Le **ColumnTransformer** de scikit-learn applique le bon traitement à chaque groupe de colonnes, et la **Pipeline** enchaîne ce prétraitement avec le modèle. C'est le cœur du chapitre « Prétraitement avec sklearn » du FullStack.

On travaille avec **Immo** : 800 ventes d'appartements, avec des valeurs manquantes et deux erreurs de saisie, comme dans la vraie vie.

In [ ]:
import pandas as pd
from immo import charger

df = charger()
print(df.shape)
print(df.isna().sum()[lambda s: s > 0])
df.head()

In [ ]:
df["prix_m2"] = df["prix"] / df["surface"]
df.sort_values("prix_m2", ascending=False).head(3)[["surface", "quartier", "prix", "prix_m2"]]

Deux appartements à plus de 50 000 € le m² : un zéro de trop dans le prix. On les retire avant toute modélisation.

## Les briques du prétraitement

| Transformateur | Rôle |
| --- | --- |
| `SimpleImputer(strategy="median")` | Remplace les valeurs manquantes |
| `StandardScaler()` | Centre et réduit (moyenne 0, écart-type 1) |
| `MinMaxScaler()` | Ramène entre 0 et 1 |
| `OneHotEncoder(handle_unknown="ignore")` | Une colonne 0/1 par catégorie |
| `OrdinalEncoder(categories=[...])` | Un entier par catégorie, dans un ordre choisi (utile pour le DPE, de A à G) |

## Assembler avec ColumnTransformer

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = df[df["prix_m2"] < 20000].drop(columns="prix_m2")
X, y = df.drop(columns="prix"), df["prix"]
NUM = ["surface", "pieces", "etage", "ascenseur", "annee_construction", "distance_metro_m", "balcon"]
CAT = ["quartier", "dpe"]

pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])
Xt = pre.fit_transform(X)
print(Xt.shape)
print(pre.get_feature_names_out()[:10])

## Pourquoi tout mettre dans une pipeline

Si tu calcules la médiane ou la moyenne de mise à l'échelle sur **toutes** les données avant de séparer entraînement et test, des informations du test fuient dans l'entraînement : c'est une **fuite de données**, et ton score est trop optimiste. Dans une pipeline, chaque transformateur apprend uniquement sur les données d'entraînement de chaque pli de validation croisée.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

modele = make_pipeline(pre, LinearRegression())
scores = cross_val_score(modele, X, y, cv=5, scoring="neg_mean_absolute_error")
print(f"Erreur absolue moyenne : {-scores.mean():,.0f} €".replace(",", " "))

> **À retenir** : un modèle de production = une pipeline (prétraitement + modèle) qu'on entraîne, évalue et sauvegarde comme un seul objet. C'est elle que tu déploieras derrière une API au module Ingénierie ML.

### Exercice 1 · Un DPE ordonné

Le DPE va de A (meilleur) à G (pire) : c'est une catégorie **ordonnée**. Crée `pre_ord`, un `ColumnTransformer` qui applique le traitement numérique de la leçon aux colonnes `NUM`, un `OrdinalEncoder` avec l'ordre `A` à `G` à la colonne `dpe`, et un `OneHotEncoder(handle_unknown="ignore")` à `quartier`. Puis crée `Xt = pre_ord.fit_transform(X)`.

<details><summary>Indice</summary>

OrdinalEncoder(categories=[list("ABCDEFG")]) : notez la liste de listes, une par colonne encodée.
</details>

In [ ]:
import pandas as pd
from immo import charger
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = charger()
df = df[df["prix"] / df["surface"] < 20000]          # retire les erreurs de saisie
X, y = df.drop(columns="prix"), df["prix"]
NUM = ["surface", "pieces", "etage", "ascenseur", "annee_construction", "distance_metro_m", "balcon"]
CAT = ["quartier", "dpe"]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])
from sklearn.preprocessing import OrdinalEncoder

Vérifie que l'affichage correspond exactement à la consigne.

### Exercice 2 · Une pipeline sans fuite

Crée `modele`, une pipeline qui enchaîne `pre` et une `LinearRegression`, puis calcule `mae_cv`, l'erreur absolue moyenne en validation croisée à 5 plis (un nombre positif, en euros).

<details><summary>Indice</summary>

cross_val_score(modele, X, y, cv=5, scoring="neg_mean_absolute_error") renvoie des scores négatifs : prends -scores.mean().
</details>

In [ ]:
import pandas as pd
from immo import charger
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = charger()
df = df[df["prix"] / df["surface"] < 20000]          # retire les erreurs de saisie
X, y = df.drop(columns="prix"), df["prix"]
NUM = ["surface", "pieces", "etage", "ascenseur", "annee_construction", "distance_metro_m", "balcon"]
CAT = ["quartier", "dpe"]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
assert isinstance(modele, Pipeline), "modele doit être une Pipeline (make_pipeline)."
ref = -cross_val_score(modele, X, y, cv=5, scoring="neg_mean_absolute_error").mean()
assert mae_cv > 0, "Le score neg_mean_absolute_error est négatif : prends son opposé."
assert abs(mae_cv - ref) < 1, f"mae_cv vaut {mae_cv:.0f}, j'attendais {ref:.0f}."
print("Bravo, tout est correct.")

---
## Régression avec scikit-learn

**scikit-learn** propose des dizaines de modèles avec la même interface : on crée le modèle, on appelle `fit` sur les données d'entraînement, puis `predict`. Tu as codé cette interface toi-même en semaine 3.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv("penguins.csv").dropna()
X = df[["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]]
y = df["body_mass_g"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele = LinearRegression()
modele.fit(X_train, y_train)
print("coefficients :", modele.coef_.round(2))
print("biais :", round(modele.intercept_, 1))

Chaque coefficient indique de combien de grammes la masse prédite augmente quand la variable augmente d'une unité, les autres restant fixes.

## Évaluer une régression

- **MAE** (erreur absolue moyenne) : l'erreur moyenne, dans l'unité de la cible. Facile à expliquer.
- **RMSE** (racine de l'erreur quadratique moyenne) : pénalise davantage les grosses erreurs.
- **R²** : la part de la variance expliquée par le modèle. 1 = parfait, 0 = pas mieux que prédire la moyenne.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

pred = modele.predict(X_test)
print(f"MAE  : {mean_absolute_error(y_test, pred):.0f} g")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.0f} g")
print(f"R²   : {r2_score(y_test, pred):.3f}")

Compare toujours à la baseline :

In [ ]:
baseline = np.full(len(y_test), y_train.mean())
print(f"RMSE baseline : {np.sqrt(mean_squared_error(y_test, baseline)):.0f} g")

## Visualiser les erreurs

In [ ]:
import plotly.express as px
resultats = pd.DataFrame({"réel": y_test, "prédit": pred})
px.scatter(resultats, x="réel", y="prédit", title="Prédictions contre valeurs réelles (g)")

Plus les points sont proches de la diagonale, meilleur est le modèle.

> **À retenir** : on ne calcule les métriques que sur le jeu de **test**. Un score mesuré sur les données d'entraînement est toujours trop optimiste.

### Exercice 1 · Prédire le pourboire

Entraîne une `LinearRegression` nommée `modele` qui prédit `tip` à partir de `total_bill` et `size` (split 25 %, `random_state=0`). Calcule `mae` sur le jeu de test et `mae_baseline`, la MAE obtenue en prédisant toujours la moyenne de `y_train`.

<details><summary>Indice</summary>

modele = LinearRegression().fit(X_train, y_train), puis mean_absolute_error(y_test, modele.predict(X_test)).
</details>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

tips = pd.read_csv("tips.csv")
X = tips[["total_bill", "size"]]
y = tips["tip"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from sklearn.linear_model import LinearRegression
assert isinstance(modele, LinearRegression), "modele doit être une LinearRegression."
assert hasattr(modele, "coef_"), "Le modèle n'est pas entraîné : appelle fit."
assert mae < mae_baseline, "Ton modèle devrait battre la baseline."
assert abs(mae - 0.6889) < 0.01, f"mae vaut {mae:.4f}, j'attendais environ 0.689."
print("Bravo, tout est correct.")

---
## Surapprentissage et validation croisée

Un **arbre de décision** pose une suite de questions (« sexe = femme ? », « classe ≤ 2 ? ») pour classer. Plus il est profond, plus il peut coller aux données d'entraînement, jusqu'à les apprendre par cœur.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)
t["age"] = t["age"].fillna(t["age"].median())
X = t[["pclass", "femme", "age", "fare", "sibsp", "parch"]]
y = t["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lignes = []
for profondeur in [1, 2, 3, 5, 8, 12, None]:
    arbre = DecisionTreeClassifier(max_depth=profondeur, random_state=0).fit(X_train, y_train)
    lignes.append({"profondeur": str(profondeur), "train": arbre.score(X_train, y_train),
                   "test": arbre.score(X_test, y_test)})
pd.DataFrame(lignes).round(3)

Sans limite de profondeur, l'arbre atteint presque 100 % sur l'entraînement mais pas sur le test : c'est le **surapprentissage**. L'écart entre les deux scores est le signal d'alarme.

## La validation croisée

Un seul découpage train/test dépend du hasard. La **validation croisée à k plis** (*k-fold*) découpe les données en k parts, entraîne k fois en gardant à chaque fois une part différente pour l'évaluation, puis fait la moyenne.

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0), X, y, cv=5)
print(scores.round(3), "→ moyenne", scores.mean().round(3), "± écart-type", scores.std().round(3))

## Choisir un hyperparamètre

`max_depth` est un **hyperparamètre** : on le fixe avant l'entraînement, il n'est pas appris. On le choisit par validation croisée, **jamais** sur le jeu de test, qui doit rester intact jusqu'à l'évaluation finale.

> **À retenir** : le jeu de test est comme l'examen final. Si tu t'en sers pour régler ton modèle, ton score final n'est plus fiable.

### Exercice 1 · Trouver la meilleure profondeur

Pour chaque `max_depth` de 1 à 10, calcule le score moyen en validation croisée (`cv=5`, `random_state=0` pour l'arbre) sur `X, y`. Stocke les résultats dans le dictionnaire `scores_cv` `{profondeur: score moyen}`, puis `meilleure_profondeur`.

<details><summary>Indice</summary>

Une boucle for d in range(1, 11), puis max(scores_cv, key=scores_cv.get).
</details>

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)
t["age"] = t["age"].fillna(t["age"].median())
X = t[["pclass", "femme", "age", "fare", "sibsp", "parch"]]
y = t["survived"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
ref = {d: cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X, y, cv=5).mean() for d in range(1, 11)}
assert set(scores_cv) == set(range(1, 11)), "scores_cv doit avoir les clés 1 à 10."
assert meilleure_profondeur == max(ref, key=ref.get), f"meilleure_profondeur vaut {meilleure_profondeur}, j'attendais {max(ref, key=ref.get)}."
print("Bravo, tout est correct.")

---
## Régularisation : Ridge, Lasso et recherche par grille

Avec beaucoup de variables, une régression linéaire peut donner des coefficients énormes qui s'ajustent au bruit : c'est le **surapprentissage**. La **régularisation** ajoute à l'erreur une pénalité sur la taille des coefficients.

$$
\text{Ridge (L2)} : \; \min_w \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j w_j^2
\qquad
\text{Lasso (L1)} : \; \min_w \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j |w_j|
$$

- **Ridge** rétrécit tous les coefficients, sans les annuler.
- **Lasso** annule les coefficients des variables peu utiles : il fait de la **sélection de variables**.
- **α** (*alpha*) règle la force de la pénalité : 0 = régression classique, très grand = coefficients presque nuls.

*Animation interactive « regpath » : à voir sur la plateforme web.*

Les variables doivent être **à la même échelle** (StandardScaler) : sinon la pénalité frappe davantage les variables exprimées en petites unités.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
X = rng.normal(size=(60, 8))
y = 3 * X[:, 0] - 2 * X[:, 1] + 0.5 * X[:, 2] + rng.normal(0, 1, 60)    # seules 3 variables comptent
X = StandardScaler().fit_transform(X)
for m in [LinearRegression(), Ridge(alpha=10), Lasso(alpha=0.3)]:
    print(f"{type(m).__name__:16}", m.fit(X, y).coef_.round(2))

## Choisir α : la recherche par grille

α est un **hyperparamètre** : on le choisit par validation croisée. `GridSearchCV` essaie toutes les valeurs d'une grille et garde la meilleure. Dans une pipeline, on nomme le paramètre `etape__parametre`.

In [ ]:
from immo import charger
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
import pandas as pd

df = charger(); df = df[df.prix / df.surface < 20000]
X, y = df.drop(columns="prix"), df["prix"]
NUM = ["surface", "pieces", "etage", "ascenseur", "annee_construction", "distance_metro_m", "balcon"]
pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), ["quartier", "dpe"])])
grille = GridSearchCV(make_pipeline(pre, Ridge()), {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                      cv=5, scoring="neg_mean_absolute_error")
grille.fit(X, y)
print("Meilleur alpha :", grille.best_params_, " MAE :", round(-grille.best_score_))
pd.DataFrame(grille.cv_results_)[["param_ridge__alpha", "mean_test_score"]].assign(mae=lambda d: -d.mean_test_score.round(0))

Pour une grande grille, `RandomizedSearchCV` tire des combinaisons au hasard : beaucoup plus rapide, presque aussi bon.

> **À retenir** : mets à l'échelle, régularise, et choisis α par validation croisée. Lasso si tu soupçonnes que beaucoup de variables sont inutiles, Ridge sinon ; ElasticNet combine les deux.

### Exercice 1 · Lasso trouve les bonnes variables

Sur les données générées ci-dessous (10 variables, seules quelques-unes comptent), entraîne un `Lasso(alpha=0.2)` et crée `gardees`, la liste des **indices** des variables dont le coefficient n'est pas nul.

<details><summary>Indice</summary>

lasso.coef_ contient un coefficient par variable ; garde les indices i où coef_[i] != 0.
</details>

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso

rng = np.random.default_rng(42)
X = rng.normal(size=(200, 10))
y = 4 * X[:, 0] + 2.5 * X[:, 3] - 3 * X[:, 7] + rng.normal(0, 1, 200)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert list(gardees) == [0, 3, 7], f"gardees vaut {list(gardees)} ; les variables utiles sont 0, 3 et 7."
print("Bravo, tout est correct.")

### Exercice 2 · Grid search sur une pipeline

Avec la pipeline Ridge de la leçon, lance un `GridSearchCV` sur `ridge__alpha` parmi `[0.1, 1, 10, 100, 1000]`, `cv=5`, en minimisant l'erreur absolue moyenne. Crée `meilleur_alpha` et `mae_meilleur` (positive).

<details><summary>Indice</summary>

Le nom du paramètre est ridge__alpha (nom de l'étape en minuscules, deux tirets bas). best_score_ est négatif.
</details>

In [ ]:
import pandas as pd
from immo import charger
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = charger()
df = df[df["prix"] / df["surface"] < 20000]          # retire les erreurs de saisie
X, y = df.drop(columns="prix"), df["prix"]
NUM = ["surface", "pieces", "etage", "ascenseur", "annee_construction", "distance_metro_m", "balcon"]
CAT = ["quartier", "dpe"]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
ref = GridSearchCV(make_pipeline(pre, Ridge()), {"ridge__alpha": [0.1, 1, 10, 100, 1000]}, cv=5, scoring="neg_mean_absolute_error").fit(X, y)
assert meilleur_alpha == ref.best_params_["ridge__alpha"], f"meilleur_alpha vaut {meilleur_alpha}, j'attendais {ref.best_params_['ridge__alpha']}."
assert abs(mae_meilleur + ref.best_score_) < 1, f"mae_meilleur vaut {mae_meilleur:.0f} ; attendu {-ref.best_score_:.0f} (pense au signe)."
print("Bravo, tout est correct.")

---
## Arbres de décision et forêts aléatoires

Un **arbre de décision** découpe l'espace des données par une suite de questions simples (« surface ≤ 45 m² ? »). Chaque feuille prédit la classe majoritaire (classification) ou la moyenne (régression) des exemples qui y tombent.

*Animation interactive « tree » : à voir sur la plateforme web.*

## Comment l'arbre choisit une question

À chaque nœud, il essaie toutes les variables et tous les seuils, et garde la coupe qui rend les deux groupes les plus **purs**. Pour la classification, la mesure d'impureté la plus courante est l'**indice de Gini** :

$$
G = 1 - \sum_k p_k^2
$$

où p_k est la proportion de la classe k dans le nœud. G = 0 : nœud pur. Pour la régression, on minimise la variance (l'erreur quadratique).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
import pandas as pd

p = pd.read_csv("penguins.csv").dropna()
X, y = p[["bill_length_mm", "flipper_length_mm"]], p["species"]
arbre = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)
print(export_text(arbre, feature_names=list(X.columns)))

Un arbre est **lisible**, ne demande ni mise à l'échelle ni encodage numérique compliqué, et capte les interactions. Mais seul, il **surapprend** dès qu'il est profond : il faut le limiter (`max_depth`, `min_samples_leaf`).

## La forêt aléatoire

Une **forêt aléatoire** entraîne des centaines d'arbres, chacun sur un **échantillon bootstrap** des données (tirage avec remise) et en ne considérant qu'un **sous-ensemble aléatoire de variables** à chaque coupe. Puis elle moyenne leurs prédictions (ou fait voter). Les erreurs des arbres se compensent : c'est le **bagging**. Résultat : beaucoup moins de surapprentissage, et un excellent modèle par défaut sur les données tabulaires.

In [ ]:
from immo import charger
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

df = charger(); df = df[df.prix / df.surface < 20000]
X = pd.get_dummies(df.drop(columns="prix"), columns=["quartier", "dpe"]).fillna(-1)
y = df["prix"]
foret = RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=0, n_jobs=1)
print("MAE :", round(-cross_val_score(foret, X, y, cv=3, scoring="neg_mean_absolute_error").mean()))
foret.fit(X, y)
pd.Series(foret.feature_importances_, index=X.columns).sort_values(ascending=False).head(6).round(3)

Les **importances** indiquent quelles variables le modèle utilise le plus. Attention : elles favorisent les variables continues et ne disent rien du sens de l'effet. Pour expliquer un modèle, on utilisera plus tard l'importance par permutation ou les valeurs SHAP.

> **À retenir** : un arbre seul = lisible mais instable ; une forêt = robuste et performante sans réglage fin. Les arbres n'ont besoin ni de mise à l'échelle, ni de linéarité.

### Exercice 1 · L'indice de Gini

Écris `gini(etiquettes)` qui renvoie l'impureté de Gini d'une liste d'étiquettes (n'importe quelles valeurs). Une liste vide a une impureté de 0.

<details><summary>Indice</summary>

Compte les classes avec Counter, puis 1 - somme des proportions au carré.
</details>

In [ ]:
from collections import Counter

def gini(etiquettes):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(gini(["a", "a", "b", "b"]) - 0.5) < 1e-9, "Deux classes à égalité : Gini = 0.5."
assert gini(["x"] * 7) == 0, "Un nœud pur a un Gini de 0."
assert abs(gini([1, 2, 3]) - 2 / 3) < 1e-9, "Trois classes équiprobables : Gini = 2/3."
assert gini([]) == 0, "Une liste vide doit renvoyer 0."
print("Bravo, tout est correct.")

### Exercice 2 · Trouver la meilleure coupe

Écris `meilleur_seuil(x, y)` qui essaie comme seuils toutes les valeurs **distinctes** de `x` (sauf la plus grande), coupe en `x <= seuil` et `x > seuil`, et renvoie le couple `(seuil, impurete)` qui minimise l'**impureté de Gini pondérée** : (n_gauche × G_gauche + n_droite × G_droite) / n. Réutilise ta fonction `gini`.

<details><summary>Indice</summary>

Boucle sur sorted(set(x))[:-1]. Pour chaque seuil, construis les deux groupes d'étiquettes et calcule l'impureté pondérée.
</details>

In [ ]:
from collections import Counter

def gini(etiquettes):
    pass   # recopie ta fonction

def meilleur_seuil(x, y):
    pass

x = [1, 2, 3, 4, 5, 6]
y = ["a", "a", "a", "b", "b", "a"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
s, imp = meilleur_seuil([1, 2, 3, 4, 5, 6], ["a", "a", "a", "b", "b", "a"])
assert s == 3 and abs(imp - 2 / 9) < 1e-9, f"meilleur_seuil renvoie {(s, imp)} ; attendu (3, 2/9)."
s2, imp2 = meilleur_seuil([10, 1, 7, 3], [1, 0, 1, 0])
assert s2 == 3 and imp2 == 0, f"Avec des x non triés : {(s2, imp2)} ; attendu (3, 0.0)."
print("Bravo, tout est correct.")

### Exercice 3 · Ce qui fait le prix

Entraîne une `RandomForestRegressor(n_estimators=100, random_state=0)` sur Immo (préparé comme dans la leçon avec `get_dummies` et `fillna(-1)`) et crée `importances`, une Series des importances indexée par le nom des variables, triée par ordre décroissant.

<details><summary>Indice</summary>

pd.Series(foret.feature_importances_, index=X.columns).sort_values(ascending=False)
</details>

In [ ]:
import pandas as pd
from immo import charger
from sklearn.ensemble import RandomForestRegressor

df = charger(); df = df[df.prix / df.surface < 20000]
X = pd.get_dummies(df.drop(columns="prix"), columns=["quartier", "dpe"]).fillna(-1)
y = df["prix"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import pandas as pd
assert isinstance(importances, pd.Series) and len(importances) == X.shape[1], "importances doit être une Series avec une valeur par colonne de X."
assert list(importances.values) == sorted(importances.values, reverse=True), "Trie par ordre décroissant."
assert importances.index[0] == "surface", f"La variable la plus importante devrait être surface ; tu as {importances.index[0]}."
assert abs(importances.sum() - 1) < 1e-6, "Les importances d'une forêt somment à 1 : as-tu bien utilisé feature_importances_ ?"
print("Bravo, tout est correct.")

---
## Boosting, XGBoost et méthodes d'ensemble

Le **boosting** construit lui aussi un ensemble d'arbres, mais **en séquence** : chaque nouvel arbre, petit, corrige les erreurs de tous les précédents. C'est la famille de modèles qui gagne la plupart des compétitions sur données tabulaires : XGBoost, LightGBM, CatBoost, et `HistGradientBoosting` dans scikit-learn.

*Animation interactive « boost » : à voir sur la plateforme web.*

## Le principe, en régression

1. On part d'une prédiction constante : la moyenne.
2. On calcule les **résidus** (valeur réelle − prédiction).
3. On entraîne un petit arbre à prédire ces résidus.
4. On ajoute ses prédictions, multipliées par un **taux d'apprentissage** η (par exemple 0,1).
5. On recommence des centaines de fois.

C'est une descente de gradient… dans l'espace des fonctions : chaque arbre est un pas dans la direction qui réduit l'erreur.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(1)
x = np.sort(rng.uniform(0, 6, 120)).reshape(-1, 1)
y = np.sin(x[:, 0]) * 3 + rng.normal(0, 0.4, 120)

pred = np.full(len(y), y.mean())
for etape in range(1, 101):
    arbre = DecisionTreeRegressor(max_depth=2).fit(x, y - pred)   # on apprend les résidus
    pred += 0.1 * arbre.predict(x)
    if etape in [1, 10, 50, 100]:
        print(f"après {etape:3} arbres : MSE = {((y - pred) ** 2).mean():.3f}")

## Bagging contre boosting

| | Forêt aléatoire (bagging) | Gradient boosting |
| --- | --- | --- |
| Arbres | Profonds, indépendants, en parallèle | Petits, en séquence |
| Réduit surtout | La variance | Le biais |
| Réglages | Peu sensibles | Taux d'apprentissage, nombre d'arbres, profondeur |
| Risque | Faible | Surapprentissage si trop d'arbres : on utilise l'arrêt anticipé |

## En pratique

In [ ]:
import pandas as pd
from immo import charger
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

df = charger(); df = df[df.prix / df.surface < 20000]
X = df.drop(columns="prix").astype({"quartier": "category", "dpe": "category"})
y = df["prix"]
hgb = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=300, categorical_features="from_dtype", random_state=0)
print("MAE :", round(-cross_val_score(hgb, X, y, cv=3, scoring="neg_mean_absolute_error").mean()))

`HistGradientBoosting` gère seul les valeurs manquantes et les catégories. Sur ton ordinateur, XGBoost s'utilise de la même façon :

```python
from xgboost import XGBRegressor      # pip install xgboost
modele = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                      subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=30)
modele.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
```

## Voter et empiler

- **Vote** (`VotingClassifier`, `VotingRegressor`) : on moyenne les prédictions de modèles différents.
- **Empilement** (*stacking*, `StackingRegressor`) : un modèle final apprend à combiner les prédictions des autres.

> **À retenir** : sur des données tabulaires, commence par une forêt aléatoire puis un gradient boosting. Règle le taux d'apprentissage et le nombre d'arbres ensemble, avec arrêt anticipé.

### Exercice 1 · Ton propre gradient boosting

Écris `boosting(x, y, n_arbres, eta, profondeur=1)` qui implémente l'algorithme de la leçon et renvoie le couple `(pred, mses)` : les prédictions finales sur `x` et la liste des MSE d'entraînement **après chaque arbre** (longueur `n_arbres`).

<details><summary>Indice</summary>

pred commence à y.mean(). À chaque tour : un arbre sur (x, y - pred), puis pred += eta * arbre.predict(x).
</details>

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(3)
x = np.sort(rng.uniform(0, 6, 150)).reshape(-1, 1)
y = np.sin(x[:, 0]) * 3 + rng.normal(0, 0.4, 150)

def boosting(x, y, n_arbres, eta, profondeur=1):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
pred, mses = boosting(x, y, 200, 0.1)
assert len(mses) == 200 and pred.shape == y.shape, "boosting doit renvoyer les prédictions et une MSE par arbre."
assert all(b <= a + 1e-12 for a, b in zip(mses, mses[1:])), "La MSE d'entraînement ne doit jamais augmenter d'un arbre à l'autre."
assert mses[-1] < 0.25 * y.var(), f"MSE finale {mses[-1]:.3f} : trop élevée, le modèle n'apprend pas les résidus."
p1, m1 = boosting(x, y, 1, 1.0)
assert abs(m1[0] - ((y - p1) ** 2).mean()) < 1e-9, "Chaque MSE doit être calculée après l'ajout de l'arbre correspondant."
print("Bravo, tout est correct.")

### Exercice 2 · Forêt contre boosting

Compare en validation croisée à 3 plis l'erreur absolue moyenne d'une `RandomForestRegressor(n_estimators=100, random_state=0)` et d'un `HistGradientBoostingRegressor(random_state=0)` sur `X_num` (variables encodées, fournies). Range les deux MAE (positives) dans le dictionnaire `maes` avec les clés `"foret"` et `"boosting"`, puis crée `meilleur`, la clé du modèle à l'erreur la plus basse.

<details><summary>Indice</summary>

Un dictionnaire de modèles, puis une compréhension de dictionnaire avec cross_val_score. min(maes, key=maes.get) donne la meilleure clé.
</details>

In [ ]:
import pandas as pd
from immo import charger
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

df = charger(); df = df[df.prix / df.surface < 20000]
X_num = pd.get_dummies(df.drop(columns="prix"), columns=["quartier", "dpe"]).fillna(-1)
y = df["prix"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert set(maes) == {"foret", "boosting"}, f"maes doit avoir les clés foret et boosting ; tu as {set(maes)}."
assert all(20000 < v < 60000 for v in maes.values()), f"Les MAE doivent être positives et de l'ordre de 30 000 € : {maes}."
assert meilleur == min(maes, key=maes.get) == "boosting", f"meilleur vaut {meilleur!r}."
print("Bravo, tout est correct.")

---
## Séries chronologiques et prévision

Une **série chronologique** est une suite de mesures dans le temps : ventes quotidiennes, trafic d'un site, consommation électrique. Elle a souvent une **tendance**, une ou plusieurs **saisonnalités** (hebdomadaire, annuelle) et du bruit. Sa particularité : l'ordre compte, et on ne doit jamais utiliser le futur pour prédire le passé.

On simule deux ans de locations quotidiennes de vélos :

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

rng = np.random.default_rng(0)
jours = pd.date_range("2024-01-01", "2025-12-31", freq="D")
t = np.arange(len(jours))
semaine = np.where(jours.dayofweek >= 5, 1.35, 1.0)                     # plus de locations le week-end
annee = 1 + 0.45 * np.sin(2 * np.pi * (jours.dayofyear - 110) / 365)    # pic en été
velos = pd.Series((200 + 0.15 * t) * semaine * annee + rng.normal(0, 18, len(t)), index=jours, name="locations").round()
px.line(x=velos.index, y=velos.values, title="Locations quotidiennes de vélos", labels={"x": "date", "y": "locations"})

## Manipuler le temps avec pandas

In [ ]:
print(velos["2025-07"].head(3))                         # sélectionner un mois
mensuel = velos.resample("MS").sum()                    # agréger par mois
lisse = velos.rolling(28, center=True).mean()           # moyenne mobile sur 28 jours
retard = velos.shift(7)                                 # la valeur d'il y a 7 jours
pd.DataFrame({"locations": velos, "il_y_a_7_jours": retard}).dropna().head(3)

*Animation interactive « ts » : à voir sur la plateforme web.*

## Évaluer une prévision

On sépare **dans le temps** : on entraîne sur le passé, on teste sur la période qui suit. Jamais de mélange aléatoire.

Et on compare toujours à deux **baselines** :
- **naïve** : demain = aujourd'hui ;
- **saisonnière naïve** : demain = le même jour la semaine dernière.

In [ ]:
test = velos["2025-11-01":]
naif = velos.shift(1)[test.index]
saisonnier = velos.shift(7)[test.index]
mae = lambda p: (test - p).abs().mean()
print(f"MAE naïve : {mae(naif):.1f}   MAE saisonnière : {mae(saisonnier):.1f}")

## Un modèle avec des variables de retard

On transforme la série en tableau classique : chaque jour devient une ligne avec, comme variables, les valeurs passées (*lags*) et le calendrier. N'importe quel modèle de régression s'applique alors. Ici on prévoit **le lendemain** : on a donc le droit d'utiliser la valeur de la veille (`lag_1`) et celles des semaines précédentes, que le modèle moyenne pour lisser le bruit.

In [ ]:
from sklearn.linear_model import LinearRegression

d = pd.DataFrame({"y": velos})
for k in [1, 7, 14, 21, 28]:
    d[f"lag_{k}"] = d["y"].shift(k)
d = d.join(pd.get_dummies(d.index.dayofweek, prefix="jour").set_index(d.index).astype(int)).dropna()
train, test_d = d[:"2025-10-31"], d["2025-11-01":]
modele = LinearRegression().fit(train.drop(columns="y"), train["y"])
pred = modele.predict(test_d.drop(columns="y"))
print(f"MAE du modèle : {(test_d['y'] - pred).abs().mean():.1f}")

## Prophet

Le FullStack utilise **Prophet** (Meta), qui décompose automatiquement tendance, saisonnalités et jours fériés. Sur ton ordinateur :

```python
from prophet import Prophet          # pip install prophet
df_p = velos.reset_index().rename(columns={"index": "ds", "locations": "y"})
m = Prophet(weekly_seasonality=True, yearly_seasonality=True).fit(df_p[df_p.ds < "2025-11-01"])
futur = m.make_future_dataframe(periods=61)
prevision = m.predict(futur)          # colonnes yhat, yhat_lower, yhat_upper
m.plot_components(prevision)
```

> **À retenir** : sépare dans le temps, bats d'abord la baseline saisonnière, et transforme la série en tableau avec des retards pour utiliser n'importe quel modèle de ML.

### Exercice 1 · Les deux baselines

Sur la série `velos` (fournie), pour la période de test à partir du 1er novembre 2025, calcule `mae_naif` (prévision = valeur de la veille) et `mae_saisonnier` (prévision = valeur 7 jours avant).

<details><summary>Indice</summary>

velos.shift(1) décale la série d'un jour ; sélectionne ensuite les dates de test avec [test.index].
</details>

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)
jours = pd.date_range("2024-01-01", "2025-12-31", freq="D")
t = np.arange(len(jours))
velos = pd.Series((200 + 0.15 * t) * np.where(jours.dayofweek >= 5, 1.35, 1.0)
                  * (1 + 0.45 * np.sin(2 * np.pi * (jours.dayofyear - 110) / 365))
                  + rng.normal(0, 18, len(t)), index=jours).round()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
test = velos["2025-11-01":]
assert abs(mae_naif - (test - velos.shift(1)[test.index]).abs().mean()) < 1e-6, "mae_naif ne correspond pas : prévision = velos.shift(1)."
assert abs(mae_saisonnier - (test - velos.shift(7)[test.index]).abs().mean()) < 1e-6, "mae_saisonnier ne correspond pas : prévision = velos.shift(7)."
assert mae_saisonnier < mae_naif, "Avec une forte saisonnalité hebdomadaire, la baseline saisonnière doit être meilleure."
print("Bravo, tout est correct.")

### Exercice 2 · Battre la baseline

Construis le tableau `d` avec la cible `y`, les retards `lag_1`, `lag_7`, `lag_14`, `lag_21`, `lag_28` et les indicatrices du jour de la semaine, comme dans la leçon. Entraîne une `LinearRegression` jusqu'au 31 octobre 2025 et calcule `mae_modele` sur la période à partir du 1er novembre. Elle doit être meilleure que la baseline saisonnière.

<details><summary>Indice</summary>

Reprends le code de la leçon. La séparation se fait par dates : d[:"2025-10-31"] et d["2025-11-01":].
</details>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(0)
jours = pd.date_range("2024-01-01", "2025-12-31", freq="D")
t = np.arange(len(jours))
velos = pd.Series((200 + 0.15 * t) * np.where(jours.dayofweek >= 5, 1.35, 1.0)
                  * (1 + 0.45 * np.sin(2 * np.pi * (jours.dayofyear - 110) / 365))
                  + rng.normal(0, 18, len(t)), index=jours).round()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert {"lag_1", "lag_7", "lag_14", "lag_21", "lag_28"} <= set(d.columns), "d doit contenir les colonnes lag_1, lag_7, lag_14, lag_21 et lag_28."
assert d.index.min() >= pd.Timestamp("2024-01-29"), "Supprime les lignes incomplètes (dropna) créées par les retards."
tst = velos["2025-11-01":]
saison = (tst - velos.shift(7)[tst.index]).abs().mean()
assert mae_modele < saison, f"mae_modele = {mae_modele:.1f} ne bat pas la baseline saisonnière ({saison:.1f})."
print("Bravo, tout est correct.")

---
## K-means et clustering

Le **clustering** regroupe des données **sans étiquettes** : segmentation de clients, regroupement de documents, compression d'images. **K-means** est l'algorithme le plus utilisé : il cherche k centres tels que chaque point soit proche du centre de son groupe.

*Animation interactive « kmeans » : à voir sur la plateforme web.*

## L'algorithme

1. Choisir k centres au départ (au hasard parmi les points, ou avec l'initialisation *k-means++*).
2. **Affecter** chaque point au centre le plus proche.
3. **Mettre à jour** chaque centre : la moyenne des points qui lui sont affectés.
4. Répéter 2 et 3 jusqu'à ce que plus rien ne bouge.

K-means minimise l'**inertie** : la somme des distances au carré entre chaque point et son centre.

$$
\text{inertie} = \sum_i \lVert x_i - \mu_{c(i)} \rVert^2
$$

In [ ]:
import numpy as np

rng = np.random.default_rng(2)
X = np.vstack([rng.normal(c, 0.6, size=(50, 2)) for c in [(0, 0), (4, 1), (2, 4)]])
centres = X[rng.choice(len(X), 3, replace=False)]
for etape in range(6):
    distances = ((X[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)   # (150, 3)
    labels = distances.argmin(axis=1)
    centres = np.array([X[labels == k].mean(axis=0) for k in range(3)])
    print(f"étape {etape}: inertie = {distances.min(axis=1).sum():.1f}")

## Avec scikit-learn, et comment choisir k

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

p = pd.read_csv("penguins.csv").dropna()
Xp = StandardScaler().fit_transform(p[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]])
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xp)
    print(f"k = {k}  inertie = {km.inertia_:7.1f}  silhouette = {silhouette_score(Xp, km.labels_):.3f}")

- La **méthode du coude** : l'inertie baisse toujours quand k augmente ; on cherche le point où elle cesse de baisser fortement.
- Le **score de silhouette** (entre −1 et 1) mesure si chaque point est plus proche de son groupe que du groupe voisin. Plus il est haut, mieux c'est.

Les manchots ont 3 espèces : K-means les retrouve-t-il sans les connaître ?

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xp)
pd.crosstab(km.labels_, p["species"])

## Les limites

- Il faut **mettre à l'échelle** : sinon la masse en grammes écrase la longueur du bec en millimètres.
- K-means suppose des groupes **ronds** et de taille comparable. Pour des formes allongées ou du bruit, on utilise DBSCAN ou un mélange de gaussiennes.
- Le résultat dépend de l'initialisation : `n_init=10` relance l'algorithme 10 fois et garde le meilleur.

> **À retenir** : mets à l'échelle, essaie plusieurs k, regarde le coude et la silhouette, puis **interprète** chaque groupe avec les moyennes de ses variables. Un clustering n'a de valeur que s'il est utile au métier.

### Exercice 1 · Affecter les points

Écris `affecter(X, centres)` qui renvoie, pour chaque point de `X` (tableau (n, d)), l'indice du centre le plus proche parmi `centres` (tableau (k, d)), **sans boucle** Python, grâce au broadcasting.

<details><summary>Indice</summary>

X[:, None, :] - centres[None, :, :] a la forme (n, k, d). Somme sur l'axe 2, puis argmin sur l'axe 1.
</details>

In [ ]:
import numpy as np

def affecter(X, centres):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
X = np.array([[0, 0], [10, 10], [1, 0], [9, 11], [6, 6]], dtype=float)
C = np.array([[0, 0], [10, 10]], dtype=float)
assert list(affecter(X, C)) == [0, 1, 0, 1, 1], f"affecter renvoie {list(affecter(X, C))}."
rng = np.random.default_rng(0); A = rng.normal(size=(200, 3)); B = rng.normal(size=(4, 3))
ref = np.array([np.argmin([((a - b) ** 2).sum() for b in B]) for a in A])
assert (affecter(A, B) == ref).all(), "Le résultat est faux en dimension 3 : vérifie les axes du broadcasting."
print("Bravo, tout est correct.")

### Exercice 2 · K-means de zéro

Écris `kmeans(X, k, n_iter=50, graine=0)` : initialise les centres avec `k` points tirés sans remise (`np.random.default_rng(graine).choice(len(X), k, replace=False)`), puis alterne affectation et mise à jour. Arrête-toi si les centres ne bougent plus. Renvoie `(labels, centres, inertie)`.

<details><summary>Indice</summary>

Boucle : affecter, calculer les nouveaux centres (moyenne par groupe), comparer avec np.allclose pour s'arrêter.
</details>

In [ ]:
import numpy as np

def affecter(X, centres):
    pass   # recopie ta fonction

def kmeans(X, k, n_iter=50, graine=0):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
rng = np.random.default_rng(2)
X = np.vstack([rng.normal(c, 0.6, size=(50, 2)) for c in [(0, 0), (4, 1), (2, 4)]])
labels, centres, inertie = kmeans(X, 3)
assert centres.shape == (3, 2) and len(labels) == 150, "kmeans doit renvoyer 150 labels et 3 centres en 2D."
for j in range(3):
    assert np.allclose(centres[j], X[labels == j].mean(axis=0)), "À la convergence, chaque centre est la moyenne de ses points."
assert abs(inertie - ((X - centres[labels]) ** 2).sum()) < 1e-6, "L'inertie doit être la somme des distances au carré à son centre."
best = KMeans(3, n_init=10, random_state=0).fit(X).inertia_
assert inertie < 1.2 * best, f"Inertie {inertie:.1f} bien supérieure à l'optimum ({best:.1f}) : l'algorithme ne converge pas."
print("Bravo, tout est correct.")

### Exercice 3 · Segmenter les manchots

Mets à l'échelle les 4 mesures des manchots, applique `KMeans(n_clusters=3, n_init=10, random_state=0)`, puis calcule `purete` : pour chaque groupe, le nombre de manchots de l'espèce majoritaire, additionné sur les 3 groupes et divisé par le nombre total de manchots.

<details><summary>Indice</summary>

pd.crosstab(labels, espèces) donne le tableau groupes × espèces ; .max(axis=1).sum() additionne les effectifs majoritaires.
</details>

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

p = pd.read_csv("penguins.csv").dropna()
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
Xr = StandardScaler().fit_transform(p[cols])
ref = pd.crosstab(KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xr).labels_, p["species"]).max(axis=1).sum() / len(p)
assert abs(purete - ref) < 1e-9, f"purete vaut {purete:.3f}, j'attendais {ref:.3f}. As-tu bien mis à l'échelle ?"
assert purete > 0.85, "Sans mise à l'échelle, la masse écrase les autres variables."
print("Bravo, tout est correct.")

---
## Projet final · Classifier les espèces de manchots

Ton premier projet de portfolio complet : à partir des mesures d'un manchot, prédire son espèce (Adelie, Chinstrap ou Gentoo). Tu mobilises toute la mise à niveau : nettoyage, exploration, modèle, évaluation, et présentation.

Une **pipeline** scikit-learn enchaîne les étapes de préparation et le modèle en un seul objet. Elle évite les erreurs, en particulier les fuites de données : la standardisation n'apprend que sur le jeu d'entraînement.

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modele.fit(X_train, y_train)
```

Modèle de README pour GitHub (à rédiger en anglais) :

```markdown
# Penguin species classifier
## Problem       What we predict and why it matters
## Data          Palmer Penguins: 344 rows, 7 columns, cleaning choices
## Approach      Features, model, train/test split
## Results       Accuracy, confusion matrix, baseline comparison
## How to run    pip install -r requirements.txt, then open the notebook
```

**À faire :**

- [ ] Refaire le projet dans un notebook sur ton ordinateur
- [ ] Ajouter un graphique exploratoire (nageoire contre bec, coloré par espèce)
- [ ] Comparer avec un arbre de décision et expliquer ton choix
- [ ] Écrire le README en anglais à partir du modèle ci-dessus
- [ ] Publier sur GitHub et épingler le dépôt sur ton profil

### Exercice 1 · Préparer les données

Charge `penguins.csv`, supprime les lignes incomplètes, et crée `X` avec les 4 mesures numériques plus une colonne `male` (1 si `sex == "MALE"`, sinon 0), et `y` = `species`. Sépare en train/test (20 %, `random_state=42`, stratifié).

<details><summary>Indice</summary>

dropna(), puis (df["sex"] == "MALE").astype(int). N'oublie pas stratify=y.
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert X.shape == (333, 5), f"X a la forme {X.shape}, j'attendais (333, 5)."
assert "male" in X.columns and set(X["male"].unique()) == {0, 1}, "La colonne male doit contenir des 0 et des 1."
assert len(X_test) == 67, f"X_test a {len(X_test)} lignes, j'en attendais 67."
print("Bravo, tout est correct.")

### Exercice 2 · Entraîner et évaluer

Crée `modele`, une pipeline `StandardScaler` + `LogisticRegression(max_iter=1000)`, entraîne-la, calcule `acc` sur le test et `cm`, la matrice de confusion. Vise une exactitude d'au moins 0.97.

<details><summary>Indice</summary>

make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)) s'utilise comme un modèle : fit puis predict.
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("penguins.csv").dropna()
df["male"] = (df["sex"] == "MALE").astype(int)
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "male"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert hasattr(modele, "predict") and hasattr(modele, "steps"), "modele doit être une pipeline (make_pipeline)."
assert acc >= 0.97, f"acc vaut {acc:.3f}, vise au moins 0.97."
assert np.asarray(cm).shape == (3, 3), "cm doit être une matrice 3 × 3 (une ligne et une colonne par espèce)."
print("Bravo, tout est correct.")

### Exercice 3 · Prédire un nouveau manchot

Utilise ta pipeline (recopie le code nécessaire) pour prédire l'espèce d'un manchot mâle avec un bec de 50 mm de long et 15 mm d'épaisseur, une nageoire de 220 mm et une masse de 5 500 g. Stocke le nom de l'espèce dans `espece` et la probabilité associée dans `confiance`.

<details><summary>Indice</summary>

Le DataFrame doit avoir exactement les mêmes colonnes que X. predict(...)[0] donne l'espèce, predict_proba(...).max() la probabilité.
</details>

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("penguins.csv").dropna()
df["male"] = (df["sex"] == "MALE").astype(int)
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "male"]
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(df[cols], df["species"])

nouveau = pd.DataFrame([{
    # à compléter
}])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert espece == "Gentoo", f"espece vaut {espece!r}."
assert 0.9 < confiance <= 1, f"confiance vaut {confiance}, elle devrait être une probabilité élevée."
print("Bravo, tout est correct.")

---
## Projet · Estimer le prix d'un appartement

L'équivalent du projet « Marché immobilier » du FullStack : construire, comparer et évaluer proprement des modèles de prix sur le jeu de données Immo, puis préparer le modèle pour la mise en production.

La démarche professionnelle, dans l'ordre :

1. Nettoyer et justifier chaque choix.
2. Fixer une **baseline**.
3. Comparer plusieurs modèles **en validation croisée**, sur le jeu d'entraînement seulement.
4. Évaluer **une seule fois** le modèle retenu sur le jeu de test.
5. Expliquer les erreurs et les limites.

**À faire :**

- [ ] Refaire le projet dans un notebook, avec une analyse exploratoire illustrée
- [ ] Ajouter un GridSearchCV sur learning_rate et max_iter du boosting
- [ ] Analyser les 10 plus grosses erreurs : quel quartier, quelle surface ?
- [ ] Sauvegarder la pipeline avec joblib.dump(modele, "modele.joblib")
- [ ] Publier sur GitHub avec un README en anglais (résultats en haut, baseline comparée)

### Exercice 1 · Nettoyer et séparer

Charge Immo, retire les ventes dont le prix au m² dépasse 20 000 €, puis sépare en `X_train, X_test, y_train, y_test` (20 % de test, `random_state=42`), avec `y` = `prix` et `X` = toutes les autres colonnes.

<details><summary>Indice</summary>

df[df["prix"] / df["surface"] <= 20000], puis train_test_split(X, y, test_size=0.2, random_state=42).
</details>

In [ ]:
import pandas as pd
from immo import charger
from sklearn.model_selection import train_test_split

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert len(X_train) + len(X_test) == 798, f"Il doit rester 798 ventes après nettoyage ; tu en as {len(X_train) + len(X_test)}."
assert "prix" not in X_train.columns, "La colonne prix ne doit pas être dans X."
assert len(X_test) == 160, f"X_test doit avoir 160 lignes ; il en a {len(X_test)}."
print("Bravo, tout est correct.")

### Exercice 2 · Comparer et choisir

Sur `X_train`, compare en validation croisée (5 plis, erreur absolue moyenne) trois pipelines : `pre` + `Ridge(alpha=10)`, `pre` + `RandomForestRegressor(n_estimators=100, random_state=0)`, et `pre` + `HistGradientBoostingRegressor(random_state=0)`. Range les MAE positives dans `scores` (clés `"ridge"`, `"foret"`, `"boosting"`), ajoute la clé `"baseline"` (MAE obtenue en prédisant toujours la moyenne de `y_train`, calculée sur `y_train`), et crée `retenu`, la clé du meilleur modèle hors baseline.

<details><summary>Indice</summary>

Un dictionnaire de modèles, une compréhension avec make_pipeline(pre, m) et cross_val_score sur X_train seulement.
</details>

In [ ]:
import pandas as pd
from immo import charger
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

df = charger(); df = df[df.prix / df.surface <= 20000]
X, y = df.drop(columns="prix"), df["prix"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
NUM = ["surface", "pieces", "etage", "ascenseur", "annee_construction", "distance_metro_m", "balcon"]
pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["quartier", "dpe"])])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert set(scores) == {"ridge", "foret", "boosting", "baseline"}, f"Clés de scores : {set(scores)}."
assert abs(scores["baseline"] - (y_train - y_train.mean()).abs().mean()) < 1, "La baseline prédit toujours la moyenne de y_train."
assert all(scores[k] < scores["baseline"] / 2 for k in ["ridge", "foret", "boosting"]), "Les trois modèles doivent largement battre la baseline : vérifie le signe des scores."
assert retenu == min(["ridge", "foret", "boosting"], key=scores.get) == "boosting", f"retenu vaut {retenu!r}."
print("Bravo, tout est correct.")

### Exercice 3 · L'évaluation finale

Entraîne la pipeline `pre` + `HistGradientBoostingRegressor(random_state=0)` sur tout `X_train`, puis calcule sur `X_test` : `mae_test` (erreur absolue moyenne en euros) et `erreur_mediane_pct` (la médiane de l'erreur absolue **en pourcentage du prix réel**).

<details><summary>Indice</summary>

Entraîne sur X_train, prédis sur X_test. L'erreur en pourcentage vaut |y_test − pred| / y_test × 100 ; prends .median().
</details>

In [ ]:
import numpy as np
import pandas as pd
from immo import charger
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

df = charger(); df = df[df.prix / df.surface <= 20000]
X, y = df.drop(columns="prix"), df["prix"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
NUM = ["surface", "pieces", "etage", "ascenseur", "annee_construction", "distance_metro_m", "balcon"]
pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["quartier", "dpe"])])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
ref = make_pipeline(pre, HistGradientBoostingRegressor(random_state=0)).fit(X_train, y_train).predict(X_test)
assert abs(mae_test - np.abs(y_test - ref).mean()) < 1, f"mae_test vaut {mae_test:.0f}, j'attendais {np.abs(y_test - ref).mean():.0f}."
assert abs(erreur_mediane_pct - (np.abs(y_test - ref) / y_test * 100).median()) < 0.01, "erreur_mediane_pct : médiane de |réel − prédit| / réel × 100."
assert erreur_mediane_pct < 12, "L'erreur médiane devrait être sous 12 %."
print("Bravo, tout est correct.")